# Notebook 05 — INT8 Quantization and Benchmark

Export YOLOv8n to CoreML with INT8 quantization and benchmark FP32 vs INT8 latency and accuracy on Apple Silicon.

In [ ]:
from ultralytics import YOLO
import coremltools as ct
import numpy as np
import time
from pathlib import Path

print(f'coremltools: {ct.__version__}')

## 1 — Export INT8 Model

FP32 (default): weights stored as 32-bit floats — full precision, 6.5 MB  
INT8: weights quantized to 8-bit integers — 4× smaller, faster on Apple Neural Engine

In [ ]:
model = YOLO('yolov8n.pt')

# Export INT8 — quantizes weights from FP32 to INT8
model.export(format='coreml', nms=True, int8=True)
print('INT8 export complete.')

In [ ]:
import os

fp32_path = 'yolov8n.mlpackage'
int8_path = 'yolov8n_int8.mlpackage'

# Rename exported model to distinguish from FP32
if Path('yolov8n.mlpackage').exists() and not Path(int8_path).exists():
    os.rename('yolov8n.mlpackage', int8_path)

fp32_size = sum(f.stat().st_size for f in Path(fp32_path).rglob('*') if f.is_file()) / 1e6
int8_size = sum(f.stat().st_size for f in Path(int8_path).rglob('*') if f.is_file()) / 1e6

print(f'FP32 model size: {fp32_size:.1f} MB')
print(f'INT8 model size: {int8_size:.1f} MB')
print(f'Size reduction:  {fp32_size / int8_size:.1f}×')

## 2 — Latency Benchmark

50 runs after 10 warmup — same methodology as notebook 04.

In [ ]:
from PIL import Image
import coremltools as ct
import numpy as np

def make_pixel_buffer(path, size=640):
    img = Image.open(path).resize((size, size))
    return img

sample = make_pixel_buffer('../Sample_Images/sample_image_1.jpeg')

def benchmark(model_path, n_warmup=10, n_runs=50):
    config = ct.ComputeUnit.ALL
    model = ct.models.MLModel(model_path, compute_units=config)
    
    # Prepare input
    inp = {model.get_spec().description.input[0].name: sample}
    iou_name = model.get_spec().description.input[1].name
    conf_name = model.get_spec().description.input[2].name
    inp[iou_name] = 0.45
    inp[conf_name] = 0.25
    
    for _ in range(n_warmup):
        model.predict(inp)
    
    times = []
    for _ in range(n_runs):
        t0 = time.perf_counter()
        model.predict(inp)
        times.append((time.perf_counter() - t0) * 1000)
    
    return np.mean(times), np.std(times)

print('Benchmarking FP32...')
fp32_mean, fp32_std = benchmark(fp32_path)
print(f'FP32: {fp32_mean:.1f} ± {fp32_std:.1f} ms')

print('Benchmarking INT8...')
int8_mean, int8_std = benchmark(int8_path)
print(f'INT8: {int8_mean:.1f} ± {int8_std:.1f} ms')

print(f'Speedup: {fp32_mean / int8_mean:.2f}×')

## 3 — Accuracy Comparison

Run both models on sample images and compare detection counts and confidence scores.
INT8 quantization should show minimal accuracy loss.

In [ ]:
from ultralytics import YOLO
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from pathlib import Path

sample_images = list(Path('../Sample_Images').glob('*.jpeg'))[:3]

fp32_model = YOLO(fp32_path)
int8_model = YOLO(int8_path)

fig, axes = plt.subplots(len(sample_images), 2, figsize=(14, 5 * len(sample_images)))
if len(sample_images) == 1:
    axes = [axes]

colors = ['#3B82F6','#22C55E','#F97316','#EC4899','#A855F7','#EF4444','#14B8A6','#EAB308']

def draw_boxes(ax, img_path, results, title):
    img = Image.open(img_path)
    ax.imshow(img)
    ax.axis('off')
    ax.set_title(title, fontsize=12, fontweight='bold')
    for box in results[0].boxes:
        x1, y1, x2, y2 = box.xyxy[0].tolist()
        cls = int(box.cls[0].item())
        conf = box.conf[0].item()
        color = colors[cls % len(colors)]
        ax.add_patch(patches.Rectangle((x1,y1), x2-x1, y2-y1,
                                        linewidth=2, edgecolor=color, facecolor='none'))
        ax.text(x1, max(y1-4,0), f'{results[0].names[cls]} {conf:.0%}',
                fontsize=8, color='white', fontweight='bold',
                bbox=dict(facecolor=color, edgecolor='none', pad=1))

for i, img_path in enumerate(sample_images):
    r_fp32 = fp32_model(str(img_path), conf=0.25, verbose=False)
    r_int8 = int8_model(str(img_path), conf=0.25, verbose=False)
    draw_boxes(axes[i][0], img_path, r_fp32, f'FP32 — {len(r_fp32[0].boxes)} detections')
    draw_boxes(axes[i][1], img_path, r_int8, f'INT8 — {len(r_int8[0].boxes)} detections')
    print(f'{img_path.name}: FP32={len(r_fp32[0].boxes)} boxes, INT8={len(r_int8[0].boxes)} boxes')

plt.suptitle('FP32 vs INT8 — Detection Comparison (conf ≥ 0.25)', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../assets/int8_comparison.png', dpi=100, bbox_inches='tight')
plt.show()
print('Saved to assets/int8_comparison.png')

## 4 — Summary Table

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

models   = ['FP32', 'INT8']
sizes    = [fp32_size, int8_size]
latencies = [fp32_mean, int8_mean]

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

bars0 = axes[0].bar(models, sizes, color=['#3B82F6', '#22C55E'], width=0.4)
axes[0].set_title('Model Size (MB)', fontweight='bold')
axes[0].set_ylabel('MB')
for bar, val in zip(bars0, sizes):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
                 f'{val:.1f} MB', ha='center', fontweight='bold')

bars1 = axes[1].bar(models, latencies, color=['#3B82F6', '#22C55E'], width=0.4)
axes[1].set_title('Inference Latency (ms) — Apple Neural Engine', fontweight='bold')
axes[1].set_ylabel('ms')
for bar, val in zip(bars1, latencies):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
                 f'{val:.1f} ms', ha='center', fontweight='bold')

plt.suptitle('FP32 vs INT8 — Size and Latency', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../assets/int8_benchmark.png', dpi=100, bbox_inches='tight')
plt.show()

print(f'\nSummary:')
print(f'  FP32: {fp32_size:.1f} MB, {fp32_mean:.1f} ms')
print(f'  INT8: {int8_size:.1f} MB, {int8_mean:.1f} ms')
print(f'  Size reduction: {fp32_size/int8_size:.1f}×')
print(f'  Speedup:        {fp32_mean/int8_mean:.2f}×')